In [44]:
import pandas as pd
import numpy as np 
from scipy.sparse import hstack
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
import faiss
from sentence_transformers import SentenceTransformer

In [37]:
df = pd.read_csv('../data/final_data/df_web.csv')
df

,Unnamed: 0,title,series,author,rating,description,language,genres,bookFormat,pages,publisher,publishDate,firstPublishDate,awards,coverImg
0,0,The Hunger Games,The Hunger Games #1,Suzanne Collins,4.33,WINNING MEANS FAME AND FORTUNE.LOSING MEANS CE...,en,"['adventure', 'dystopia', 'fantasy']",Hardcover,374,Scholastic Press,2008-09-14 00:00:00,False,True,https://i.gr-assets.com/images/S/compressed.ph...
1,1,Harry Potter and the Order of the Phoenix,Harry Potter #5,"J.K. Rowling, Mary GrandPré (Illustrator)",4.50,There is a door at the end of a silent corrido...,en,"['adventure', 'childrens', 'classics']",Paperback,870,Scholastic Inc.,2004-09-28 00:00:00,True,True,https://i.gr-assets.com/images/S/compressed.ph...
2,2,To Kill a Mockingbird,To Kill a Mockingbird,Harper Lee,4.28,The unforgettable novel of a childhood in a sl...,en,"['classics', 'fiction', 'historical']",Paperback,324,Harper Perennial Modern Classics,2006-05-23 00:00:00,True,True,https://i.gr-assets.com/images/S/compressed.ph...
3,3,Pride and Prejudice,Standalone Novel,"Jane Austen, Anna Quindlen (Introduction)",4.26,Alternate cover edition of ISBN 9780679783268S...,en,"['classics', 'fiction', 'historical']",Paperback,279,Modern Library,2000-10-10 00:00:00,True,False,https://i.gr-assets.com/images/S/compressed.ph...
4,4,Twilight,The Twilight Saga #1,Stephenie Meyer,3.60,About three things I was absolutely positive.\...,en,"['fantasy', 'fiction', 'paranormal']",Paperback,501,"Little, Brown and Company",2006-09-06 00:00:00,True,True,https://i.gr-assets.com/images/S/compressed.ph...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57300,74779,Beasts & Behemoths (Dungeons & Dragons),Standalone Novel,"Jim Zub, Stacy King, Andrew Wheeler, Official ...",4.04,Study this guide and keep it close at hand--th...,und,['fiction'],Paperback,114,Ten Speed Press,2020-10-20 00:00:00,True,False,http://books.google.com/books/content?id=1toui...
57301,74780,Faculty of Dragon Riders,Standalone Novel,Dmitry Nazarov,4.04,I tamed the Black Dragon!So I thought until I ...,und,['fiction'],Paperback,401,Litres,2022-08-24 00:00:00,True,False,http://books.google.com/books/content?id=QSGFE...
57302,74786,Midnight Delivery Sex,Standalone Novel,Neneko Narazaki,4.00,Mit SNS-Card zum Sammeln in der ersten Auflage...,de,['comics'],Paperback,29,Hayabusa,2021-05-04 00:00:00,True,False,http://books.google.com/books/content?id=s_8_E...
57303,74787,Monster Girl: 2,Standalone Novel,Kazuki Funatsu,4.00,"Dopo il loro incontro, Yatsuki si ritrova ad a...",it,"['comics', 'isekai']",Paperback,216,Edizioni BD,2020-05-01 00:00:00,True,False,http://books.google.com/books/content?id=_yjnE...


In [38]:
df = df.drop('Unnamed: 0', axis =1)

In [39]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 57305 entries, 0 to 57304
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   title             57305 non-null  object 
 1   series            57305 non-null  object 
 2   author            57305 non-null  object 
 3   rating            57305 non-null  float64
 4   description       57305 non-null  object 
 5   language          57305 non-null  object 
 6   genres            57305 non-null  object 
 7   bookFormat        57305 non-null  object 
 8   pages             57305 non-null  int64  
 9   publisher         57305 non-null  object 
 10  publishDate       57305 non-null  object 
 11  firstPublishDate  57305 non-null  bool   
 12  awards            57305 non-null  bool   
 13  coverImg          57305 non-null  object 
dtypes: bool(2), float64(1), int64(1), object(10)
memory usage: 5.4+ MB


Hacemos Label Encoding de series, language y bookformat.
Los Booleanos firstpublishdate yawards los pasamos a int.
Genres lo pasamos a multilabelbinarizer.
Author hacemos la media de rating.

In [40]:
df_model = df.copy()

cat_cols = ['series', 'language', 'bookFormat']
label_encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    df_model[col] = df_model[col].astype(str).fillna('missing')
    df_model[col] = le.fit_transform(df_model[col])
    label_encoders[col] = le

df_model['firstPublishDate'] = df_model['firstPublishDate'].astype(int)
df_model['awards'] = df_model['awards'].astype(int)

author_avg = df_model.groupby('author')['rating'].mean()
df_model['author_rating'] = df_model['author'].map(author_avg)
df_model['author_rating'] = df_model['author_rating'].fillna(df_model['rating'].mean())

mlb = MultiLabelBinarizer()
df_model['genres'] = df_model['genres'].apply(lambda g: g if isinstance(g, list) else [])
genres_ohe = mlb.fit_transform(df_model['genres'])
df_genres = pd.DataFrame(genres_ohe, columns=mlb.classes_, index=df_model.index)

df_model = pd.concat([df_model, df_genres], axis=1)

In [41]:
df_model

,title,series,author,rating,description,language,genres,bookFormat,pages,publisher,publishDate,firstPublishDate,awards,coverImg,author_rating
0,The Hunger Games,16083,Suzanne Collins,4.33,WINNING MEANS FAME AND FORTUNE.LOSING MEANS CE...,14,[],35,374,Scholastic Press,2008-09-14 00:00:00,0,1,https://i.gr-assets.com/images/S/compressed.ph...,4.227143
1,Harry Potter and the Order of the Phoenix,6493,"J.K. Rowling, Mary GrandPré (Illustrator)",4.50,There is a door at the end of a silent corrido...,14,[],62,870,Scholastic Inc.,2004-09-28 00:00:00,1,1,https://i.gr-assets.com/images/S/compressed.ph...,4.570000
2,To Kill a Mockingbird,18386,Harper Lee,4.28,The unforgettable novel of a childhood in a sl...,14,[],62,324,Harper Perennial Modern Classics,2006-05-23 00:00:00,1,1,https://i.gr-assets.com/images/S/compressed.ph...,3.693333
3,Pride and Prejudice,13751,"Jane Austen, Anna Quindlen (Introduction)",4.26,Alternate cover edition of ISBN 9780679783268S...,14,[],62,279,Modern Library,2000-10-10 00:00:00,1,0,https://i.gr-assets.com/images/S/compressed.ph...,4.260000
4,Twilight,17776,Stephenie Meyer,3.60,About three things I was absolutely positive.\...,14,[],62,501,"Little, Brown and Company",2006-09-06 00:00:00,1,1,https://i.gr-assets.com/images/S/compressed.ph...,3.938750
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57300,Beasts & Behemoths (Dungeons & Dragons),13751,"Jim Zub, Stacy King, Andrew Wheeler, Official ...",4.04,Study this guide and keep it close at hand--th...,58,[],62,114,Ten Speed Press,2020-10-20 00:00:00,1,0,http://books.google.com/books/content?id=1toui...,4.040000
57301,Faculty of Dragon Riders,13751,Dmitry Nazarov,4.04,I tamed the Black Dragon!So I thought until I ...,58,[],62,401,Litres,2022-08-24 00:00:00,1,0,http://books.google.com/books/content?id=QSGFE...,4.040000
57302,Midnight Delivery Sex,13751,Neneko Narazaki,4.00,Mit SNS-Card zum Sammeln in der ersten Auflage...,11,[],62,29,Hayabusa,2021-05-04 00:00:00,1,0,http://books.google.com/books/content?id=s_8_E...,4.000000
57303,Monster Girl: 2,13751,Kazuki Funatsu,4.00,"Dopo il loro incontro, Yatsuki si ritrova ad a...",30,[],62,216,Edizioni BD,2020-05-01 00:00:00,1,0,http://books.google.com/books/content?id=_yjnE...,4.000000


Para la recomendación de llibros vamos a utilizar FAISS y all-MiniLM-L6-v2 para hacer embedding y que combine descripción y género y que al buscar libros, te haga la búsqueda sobre estas dos columnas.

In [45]:
df['genres'] = df['genres'].apply(lambda g: g if isinstance(g, list) else [])
df['text_for_embedding'] = df['description'] + ". Genres: " + df['genres'].apply(lambda g: ", ".join(g))
modelo = SentenceTransformer('all-MiniLM-L6-v2')
print("Generating embeddings...")
embeddings = modelo.encode(df['text_for_embedding'].tolist(), convert_to_tensor=False, show_progress_bar=True)
dimension = embeddings[0].shape[0]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

def recommend_books(user_text, top_n=5):
    vec = modelo.encode([user_text])
    distances, indices = index.search(np.array(vec), top_n)
    results = df.iloc[indices[0]].copy()
    results['score'] = distances[0]
    return results[['title', 'author', 'genres', 'rating', 'coverImg', 'description', 'score']]

query = "I want to read a science fiction story with philosophical themes"
results = recommend_books(query)

for i, row in results.iterrows():
    print(f"\n\033[1m{row['title']}\033[0m by {row['author']}")
    print(f"Rating: {row['rating']} | Score: {row['score']:.4f}")
    print(f"Genres: {', '.join(row['genres'])}")
    print(f"Description: {row['description'][:300]}...")
    print(f"Cover: {row['coverImg']}")

Generating embeddings...


Batches:   0%|          | 0/1791 [00:00<?, ?it/s]

c:\Users\elena\anaconda3\Lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)



The Egg by Andy Weir (Goodreads Author)
Rating: 4.19 | Score: 0.8745
Genres: 
Description: A short story about the universe....
Cover: https://i.gr-assets.com/images/S/compressed.photo.goodreads.com/books/1431492647l/17563539.jpg

Mallworld by Somtow Sucharitkul
Rating: 3.71 | Score: 0.9006
Genres: 
Description: Science fiction novel....
Cover: https://i.gr-assets.com/images/S/compressed.photo.goodreads.com/books/1387745939l/445426.jpg

Nobel Dreams: Power, Deceit, and the Ultimate Experiment by Gary Taubes (Goodreads Author)
Rating: 4.12 | Score: 0.9027
Genres: 
Description: A modern-day adventure story demonstrating the chaotic and chancy nature of science that will change perceptions of scientific research. Several colorful personalities are introduced--the Italian physicist Carlo Rubbia chief among them....
Cover: https://i.gr-assets.com/images/S/compressed.photo.goodreads.com/books/1387668146l/3071857.jpg

The Complete Stories, Vol 1 by Isaac Asimov
Rating: 4.36 | Score: 0.9284
G